In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yeonjiji/separation-data/test.jsonl
/kaggle/input/datasets/yeonjiji/separation-data/validate.jsonl
/kaggle/input/datasets/yeonjiji/separation-data/train.jsonl
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/__results__.html
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/__huggingface_repos__.json
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/__notebook__.ipynb
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/__output__.json
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/custom.css
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/separationllm-qwen3-1.7b-lora/README.md
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/separationllm-qwen3-1.7b-lora/checkpoint-603/adapter_model.safetensors
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/separationllm-qwen3-1.7b-lora/checkpoint-603/trainer_state.json
/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/separationllm

In [2]:
# Install packages required for SAE feature intervention and LoRA model loading

%pip uninstall -y torchao

%pip install -q \
    transformers==5.16.1 \
    peft==0.20.0 \
    datasets \
    accelerate \
    scikit-learn \
    tqdm

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, re, random
import numpy as np
import torch
import torch.nn as nn
from collections import Counter

# Same seed as 02/03
SEED = 2024
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Inputs from earlier notebooks (check exact paths with os.walk if a file isn't found)
ADAPTER_PATH = "/kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/separationllm-qwen3-1.7b-lora-final"
SAE_PATH     = "/kaggle/input/notebooks/yeonjiji/03-2-sae-layer19/sae_layer19_topk64.pt"
CF_PATH      = "/kaggle/input/notebooks/yeonjiji/03-2-sae-layer19/sae_counterfactual.npz"
DATA_DIR     = "/kaggle/input/datasets/yeonjiji/separation-data"

for p in [ADAPTER_PATH, SAE_PATH, CF_PATH, DATA_DIR]:
    print(os.path.exists(p), p)

True /kaggle/input/notebooks/yeonjiji/01-fine-tune-separationllm/separationllm-qwen3-1.7b-lora-final
True /kaggle/input/notebooks/yeonjiji/03-2-sae-layer19/sae_layer19_topk64.pt
True /kaggle/input/notebooks/yeonjiji/03-2-sae-layer19/sae_counterfactual.npz
True /kaggle/input/datasets/yeonjiji/separation-data


In [4]:
# same SAE class as 03 (architecture much match to load the weights)
class TopKSparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, k):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)
        self.k = k

    def forward(self, x):
        pre = torch.relu(self.encoder(x))        # scores for all features
        values, indices = torch.topk(pre, self.k, dim=-1)    # keep top-k only
        features = torch.zeros_like(pre).scatter_(1, indices, values)
        recon = self.decoder(features)        # rebuild the input vector
        return recon, features

device = "cuda" if torch.cuda.is_available() else "cpu"

# weights_only=False: the file also stores numpy arrays (mean, std, feature lists)
ckpt = torch.load(SAE_PATH, map_location=device, weights_only=False)

sae = TopKSparseAutoencoder(ckpt["input_dim"], ckpt["sae_dim"], ckpt["k"]).to(device)
sae.load_state_dict(ckpt["state_dict"])    # exact weights from 03 -> same feature numbers
sae.eval()

mean, std = ckpt["mean"], ckpt["std"]    # normalization used when the SAE was trained

LAYER = ckpt["layer"]
top_positive = ckpt["top_positive"]
top_negative = ckpt["top_negative"]
REP_HIGH = ckpt["rep_high"]
REP_LOW = ckpt["rep_low"]

print("Layer:", LAYER, "| SAE dim:", ckpt["sae_dim"], "| k:", ckpt["k"])
print("REP_HIGH:", REP_HIGH, "| REP_LOW:", REP_LOW)
print("top_positive:", top_positive)
print("top_negative:", top_negative)

Layer: 19 | SAE dim: 2048 | k: 64
REP_HIGH: 1208 | REP_LOW: 245
top_positive: [1208  380 1205 1256 1808  991  565 1490 1423  355]
top_negative: [ 827 1615  245 1014  271 1497 1627 1239 1999  402]


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset
from tqdm.auto import tqdm

MODEL_NAME = "Qwen/Qwen3-1.7B"
DEVICE = "cuda"

# base model & ft model setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="cuda")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()

# Transformer blocks. hidden_states[k] = output of layers[k-1],
# layer 19 (hidden_states[19]) is the output of layers[18]
layers = ft_model.base_model.model.model.layers
# ft_model(PeftModel: generic PEFT wrapper — save/load, adapter on/off)
#.base_model(LoraModel: LoRA-specific wrapper — inserts and manages LoRA matrices)
#.model(Qwen3ForCausalLM: Transformer body + output layer lm_head)
#.model(Qwen3Model: Transformer body)
#.layers(28 layers, each with small LoRA matrices inserted inside)

test_dataset = load_dataset("json", data_files=f"{DATA_DIR}/test.jsonl", split="train")
cf_data = np.load(CF_PATH)
cf_idx = cf_data["cf_idx"]
print("Test samples:", len(test_dataset), "| counterfactual samples:", len(cf_idx))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Test samples: 501 | counterfactual samples: 458


#### Define functions

In [6]:
LABELS = ["VL", "L", "U", "M", "H"]
LABEL_VALUES = np.array([0, 1, 2, 3, 4])
# Token id of each class label, the model next-token logits at these ids = class scores
label_token_ids = [tokenizer.encode(l, add_special_tokens=False)[0] for l in LABELS]

# Softmax over five label logits -> probabilities -> weighted average class
def expected_d_from_logits(logits):
    probs = torch.softmax(logits[label_token_ids].float(), dim=-1).cpu().numpy()
    return float(np.sum(probs * LABEL_VALUES))

def prepare(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                            add_generation_prompt=True, enable_thinking=False)
    return tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)

# Normal forward pase, returns layer19 last token vector, expected D
# use_base=True -> lora off and behaves as the base model
def run_model(messages, use_base=False):
    inputs = prepare(messages)
    ctx = ft_model.disable_adapter() if use_base else torch.no_grad()
    with ctx, torch.no_grad():
        out = ft_model(**inputs, output_hidden_states=True, use_cache=False, return_dict=True)
    act = out.hidden_states[LAYER][0, -1, :].float()
    return act, expected_d_from_logits(out.logits[0, -1])


# Forward apss where the last token vector after layer19 is replaced by new_act
# everything else in the model runs normally, returns expected D
def run_with_patch(messages, new_act, use_base=False):
    inputs = prepare(messages)

    def hook(module, inp, output):
        hidden = (output[0] if isinstance(output, tuple) else output).clone()
        hidden[0, -1, :] = new_act.to(hidden.dtype)
        return (hidden,) + output[1:] if isinstance(output, tuple) else hidden

    handle = layers[LAYER - 1].register_forward_hook(hook)    # layers[18] -> hidden_states[19]
    ctx = ft_model.disable_adapter() if use_base else torch.no_grad()
    try:
        with ctx, torch.no_grad():
            out = ft_model(**inputs, use_cache=False, return_dict=True)
    finally:
        handle.remove()
    return expected_d_from_logits(out.logits[0, -1])


mean_t = torch.tensor(mean.squeeze(), dtype=torch.float32, device=DEVICE)
std_t  = torch.tensor(std.squeeze(), dtype=torch.float32, device=DEVICE)

# layer19 vector -> SAE features [2048] 
def encode(act):
    with torch.no_grad():
        _, f = sae(((act - mean_t) / std_t).unsqueeze(0))    # normalize first
    return f[0]

# how much the layer19 vector changes when features go from f_old to f_new
# only the difference is decoded, so the SAAE's recon error is not added in
def decode_delta(f_new, f_old):
    with torch.no_grad():
        d = sae.decoder(f_new.unsqueeze(0)) - sae.decoder(f_old.unsqueeze(0))
    return d[0] * std_t    # un-normalize

# HNO3 counterfactual
def extract_conc(text, name):
    m = re.search(rf'([\d.]+)\s*M\s*{name}', text)
    return float(m.group(1)) if m else None

def make_hno3_counterfactual(example, new_hno3):
    messages = example["messages"]; text = messages[1]["content"]
    old_hno3, old_nit = extract_conc(text, "HNO3"), extract_conc(text, "nitrate")
    new_text = re.sub(r'([\d.]+)\s*M\s*HNO3', f'{new_hno3:g} M HNO3', text)
    if old_nit is not None and old_hno3 is not None:
        new_nit = new_hno3 + max(old_nit - old_hno3, 0)
        new_text = re.sub(r'([\d.]+)\s*M\s*nitrate', f'{new_nit:g} M nitrate', new_text)
    return [messages[0], {"role": "user", "content": new_text}]

msgs = test_dataset[int(cf_idx[0])]["messages"][:-1]
act, d0 = run_model(msgs)

# Check if patching in the same vector not change anything 
d_same = run_with_patch(msgs, act)
print(f"Expected D: normal = {d0:.4f} | patched with same vector = {d_same:.4f}")

# Check if features match what 03 computer for this sample
print("Active features (should be 64):", int((encode(act) > 0).sum()))

Expected D: normal = 2.1892 | patched with same vector = 2.1892
Active features (should be 64): 64


In [7]:
# Replace the whole vector with the SAE reconstruction
def full_reconstruction(act):
    with torch.no_grad():
        recon_norm, _ = sae(((act - mean_t) / std_t).unsqueeze(0))
    return recon_norm[0] * std_t + mean_t

d_orig_list, d_recon_list = [], []

for idx in tqdm(cf_idx):
    msgs = test_dataset[int(idx)]["messages"][:-1]               # orig prompt, no answer
    act, d_orig = run_model(msgs)                                # normal prediction
    d_recon = run_with_patch(msgs, full_reconstruction(act))     # prediction with sae-only vector
    d_orig_list.append(d_orig)
    d_recon_list.append(d_recon)

d_orig_arr = np.array(d_orig_list)
d_recon_arr = np.array(d_recon_list)
diff = d_recon_arr - d_orig_arr

print(f"Samples: {len(diff)}")
print(f"Mean |ΔD| from reconstruction: {np.abs(diff).mean():.3f}")
print(f"Max  |ΔD| from reconstruction: {np.abs(diff).max():.3f}")
print(f"Mean ΔD (signed):              {diff.mean():+.3f}")
print("Reference: HNO3 1M -> 6M effect in 02 ≈ +0.68")

  0%|          | 0/458 [00:00<?, ?it/s]

Samples: 458
Mean |ΔD| from reconstruction: 0.027
Max  |ΔD| from reconstruction: 0.252
Mean ΔD (signed):              -0.004
Reference: HNO3 1M -> 6M effect in 02 ≈ +0.68


#### base vs ft_model
Applying the same SAE to the base model\
1. Does the same SAE reconstruct base vectors well?
2. Does the prediction for D change with HNO3?
3. Do feature 1208 and 245 repond to HNO3 changes? 

In [8]:
def recon_quality(act):
    x = ((act - mean_t) / std_t).unsqueeze(0)
    with torch.no_grad():
        recon, _ = sae(x)
    return 1 - ((x - recon) ** 2).sum().item() / (x ** 2).sum().item()

results = {name: {"f1": [], "f6": [], "d1": [], "d6": [], "rq": []} for name in ["ft", "base"]}

for idx in tqdm(cf_idx):
    example = test_dataset[int(idx)]
    msg_1M = make_hno3_counterfactual(example, 1)
    msg_6M = make_hno3_counterfactual(example, 6)

    for name, use_base in [("ft", False), ("base", True)]:
        a1, d1 = run_model(msg_1M, use_base=use_base)
        a6, d6 = run_model(msg_6M, use_base=use_base)
        r = results[name]
        r["f1"].append(encode(a1).cpu().numpy())
        r["f6"].append(encode(a6).cpu().numpy())
        r["d1"].append(d1); r["d6"].append(d6)
        r["rq"].append(recon_quality(a1))

for r in results.values():
    for k in r:
        r[k] = np.array(r[k])

print("done:", results["ft"]["f1"].shape)

  0%|          | 0/458 [00:00<?, ?it/s]

done: (458, 2048)


In [9]:
print("=== 1) SAE reconstruction quality (higher = better fit) ===")
for name in ["ft", "base"]:
    print(f"{name:>4}: {results[name]['rq'].mean():.3f}")

print("\n=== 2) Expected D, HNO3 1 M -> 6 M ===")
for name in ["ft", "base"]:
    r = results[name]
    print(f"{name:>4}: 1M = {r['d1'].mean():.2f} | 6M = {r['d6'].mean():.2f} | Δ = {(r['d6'] - r['d1']).mean():+.2f}")

print("\n=== 3) Representative features, HNO3 1 M -> 6 M ===")
for f, label in [(REP_HIGH, "high-D"), (REP_LOW, "low-D")]:
    for name in ["ft", "base"]:
        r = results[name]
        on_rate = ((r["f1"][:, f] > 0) | (r["f6"][:, f] > 0)).mean()   # active at 1M or 6M
        print(f"Feature {f:4d} ({label}) | {name:>4}: "
              f"1M = {r['f1'][:, f].mean():.2f} | 6M = {r['f6'][:, f].mean():.2f} | "
              f"Δ = {(r['f6'][:, f] - r['f1'][:, f]).mean():+.2f} | active = {on_rate:.2f}")

print("\n=== 3b) All 10 high-D / 10 low-D features, mean Δ ===")
for name in ["ft", "base"]:
    r = results[name]
    dpos = (r["f6"][:, top_positive] - r["f1"][:, top_positive]).mean()
    dneg = (r["f6"][:, top_negative] - r["f1"][:, top_negative]).mean()
    print(f"{name:>4}: high-D Δ = {dpos:+.2f} | low-D Δ = {dneg:+.2f}")

=== 1) SAE reconstruction quality (higher = better fit) ===
  ft: 0.976
base: 0.438

=== 2) Expected D, HNO3 1 M -> 6 M ===
  ft: 1M = 1.67 | 6M = 2.48 | Δ = +0.81
base: 1M = 0.33 | 6M = 0.33 | Δ = -0.00

=== 3) Representative features, HNO3 1 M -> 6 M ===
Feature 1208 (high-D) |   ft: 1M = 0.46 | 6M = 4.92 | Δ = +4.46 | active = 0.74
Feature 1208 (high-D) | base: 1M = 0.00 | 6M = 0.00 | Δ = +0.00 | active = 0.00
Feature  245 (low-D) |   ft: 1M = 1.08 | 6M = 0.16 | Δ = -0.93 | active = 0.73
Feature  245 (low-D) | base: 1M = 0.00 | 6M = 0.00 | Δ = +0.00 | active = 0.00

=== 3b) All 10 high-D / 10 low-D features, mean Δ ===
  ft: high-D Δ = +1.45 | low-D Δ = -0.74
base: high-D Δ = +0.01 | low-D Δ = +0.00


**Base vs. fine-tuned (same SAE, same prompts)**

- The SAE reconstructs fine-tuned activations well (0.98) but base activations poorly (0.44), so layer-19 representations changed substantially **during fine-tuning**.
- The base model does not respond to HNO₃ (1 M → 6 M: Δ expected D = 0.00), while the fine-tuned model does (+0.81).
- Features 1208 and 245 are never active in the base model, and the 20 selected features show no HNO₃ response there.

The D/HNO₃ features found by the SAE are *specific to the fine-tuned model*. The base model may still encode D-related information in a different form (probe accuracy 0.57 in 02), but not along these feature directions.

#### Change the Feature

In [10]:
# Run the model with SAE features set to new values
def intervene(messages, changes, use_base=False):
    act, _ = run_model(messages, use_base=use_base)   # original layer-19 vector
    f = encode(act)                                     # its SAE features
    f_new = f.clone()
    for fid, val in changes.items():
        f_new[fid] = float(val)                         # set chosen features
    new_act = act + decode_delta(f_new, f)              # add ONLY the change (keeps recon error)
    return run_with_patch(messages, new_act, use_base=use_base)

ft = results["ft"]
v1208 = ft["f6"][:, REP_HIGH]; v1208 = v1208[v1208 > 0 ]   # 1208 when active at 6 M
v245  = ft["f1"][:, REP_LOW];  v245  = v245[v245 > 0]      # 245 when active at 1M
H_MED, H_P95 = np.median(v1208), np.percentile(v1208, 95)
L_MED, L_P95 = np.median(v245),  np.percentile(v245, 95)
print(f"Feature {REP_HIGH} natural (6 M, active): median = {H_MED:.2f}, p95 = {H_P95:.2f}")
print(f"Feature {REP_LOW} natural (1 M, active): median = {L_MED:.2f}, p95 = {L_P95:.2f}")

Feature 1208 natural (6 M, active): median = 5.88, p95 = 15.12
Feature 245 natural (1 M, active): median = 1.48, p95 = 2.17


In [11]:
# Pick a clear example: at 1 M, 1208 is off and 245 is on, and the real 1M->6M effect is large
cand = np.where((ft["f1"][:, REP_HIGH] == 0) & (ft["f1"][:, REP_LOW] > 0))[0]
s = cand[np.argmax((ft["d6"] - ft["d1"])[cand])]
example = test_dataset[int(cf_idx[s])]
msg_1M = make_hno3_counterfactual(example, 1)

print(msg_1M[1]["content"], "\n")
print(f"Real 1 M prediction: {ft['d1'][s]:.2f}")
print(f"Real 6 M prediction: {ft['d6'][s]:.2f}   <- target\n")

# Each row: which features we change, starting from the 1 M prompt
tests = [
    ("no change (check)",            {}),
    (f"{REP_HIGH} -> median",        {REP_HIGH: H_MED}),
    (f"{REP_HIGH} -> p95",           {REP_HIGH: H_P95}),
    (f"{REP_LOW} -> 0 (turn off)",   {REP_LOW: 0.0}),
    (f"{REP_HIGH} -> median + {REP_LOW} -> 0", {REP_HIGH: H_MED, REP_LOW: 0.0}),
]

for name, ch in tests:
    d = intervene(msg_1M, ch)
    print(f"{name:<32} expected D = {d:.2f} | Δ vs 1 M = {d - ft['d1'][s]:+.2f}")

1 M of CCCCC(CC)CN(CC(CC)CCCC)C(=O)CCC in TPH is used to extract 0.3 mM Pu(IV) from an aqueous solution of 1 M HNO3 and 1 M nitrate at about 20 C. The contact time is 10 min and the radiolytic dosage is 0 kGy. 

Real 1 M prediction: 1.18
Real 6 M prediction: 3.62   <- target

no change (check)                expected D = 1.18 | Δ vs 1 M = +0.00
1208 -> median                   expected D = 1.28 | Δ vs 1 M = +0.10
1208 -> p95                      expected D = 1.64 | Δ vs 1 M = +0.46
245 -> 0 (turn off)              expected D = 1.17 | Δ vs 1 M = -0.01
1208 -> median + 245 -> 0        expected D = 1.28 | Δ vs 1 M = +0.10


In [12]:
msg_6M = make_hno3_counterfactual(example, 6)

# Real SAE features of this sample at 1 M and 6 M
f1 = encode(run_model(msg_1M)[0])
f6 = encode(run_model(msg_6M)[0])

# Rank features by how much they changed between 1 M and 6 M (this sample only)
change = (f6 - f1).abs().cpu().numpy()
order = np.argsort(change)[::-1]
n_changed = int((change > 0).sum())
print(f"Features that changed 1M -> 6M: {n_changed}\n")

d1, d6 = ft["d1"][s], ft["d6"][s]

# put 6M values of these features into the 1M run
def patch_features(feature_ids):
    return intervene(msg_1M, {int(i): f6[int(i)].item() for i in feature_ids})

rows = [("1208 only", [REP_HIGH]),
        ("selected 20 (top_pos + top_neg)", list(top_positive) + list(top_negative))]

for k in [1, 5, 10, 20, 50, 100, n_changed]:
    rows.append((f"top {k} most changed", order[:k]))

for name, ids in rows:
    d = patch_features(ids)
    recovery = (d - d1) / (d6 - d1)          # 0 = like 1 M, 1 = like real 6 M
    print(f"{name:<34} D = {d:.2f} | recovery = {recovery:5.0%}")

Features that changed 1M -> 6M: 100

1208 only                          D = 1.41 | recovery =    9%
selected 20 (top_pos + top_neg)    D = 2.26 | recovery =   44%
top 1 most changed                 D = 1.41 | recovery =    9%
top 5 most changed                 D = 2.27 | recovery =   45%
top 10 most changed                D = 2.63 | recovery =   59%
top 20 most changed                D = 3.17 | recovery =   81%
top 50 most changed                D = 3.50 | recovery =   95%
top 100 most changed               D = 3.54 | recovery =   96%
top 100 most changed               D = 3.54 | recovery =   96%


In [13]:
a1, _ = run_model(msg_1M)
a6, _ = run_model(msg_6M)
r1, r6 = full_reconstruction(a1), full_reconstruction(a6)   # SAE-only versions
e1, e6 = a1 - r1, a6 - r6                                   # reconstruction errors

def rec(vec):
    d = run_with_patch(msg_1M, vec)
    return d, (d - d1) / (d6 - d1)

tests = [
    ("A) real 6M vector (like 02 patching)",   a6),        # should be ~90-100%
    ("B) 6M SAE recon only (no error)",         r6),        # features alone, error dropped
    ("C) 6M recon + 1M error (= all features)", r6 + e1),   # what cell 10 did
    ("D) 1M recon + 6M error (error only)",     r1 + e6),   # only the error is from 6M
]
for name, v in tests:
    d, r = rec(v)
    print(f"{name:<42} D = {d:.2f} | recovery = {r:5.0%}")

# How big is the error compared with the 1M->6M change itself?
print(f"\n|a6 - a1|  (whole change)   = {(a6 - a1).norm():.2f}")
print(f"|r6 - r1|  (feature part)   = {(r6 - r1).norm():.2f}")
print(f"|e6 - e1|  (error part)     = {(e6 - e1).norm():.2f}")

A) real 6M vector (like 02 patching)       D = 3.60 | recovery =   99%
B) 6M SAE recon only (no error)            D = 3.57 | recovery =   98%
C) 6M recon + 1M error (= all features)    D = 3.54 | recovery =   96%
D) 1M recon + 6M error (error only)        D = 1.23 | recovery =    2%

|a6 - a1|  (whole change)   = 120.66
|r6 - r1|  (feature part)   = 113.68
|e6 - e1|  (error part)     = 19.56


In [14]:
# Check and find bug
# Path 1: build the vector directly (should equal C exactly)
v_direct = a1 + decode_delta(f6, f1)
print("max |v_direct - C|:", (v_direct - (r6 + e1)).abs().max().item())   # expect ~0
print("D with v_direct:", round(run_with_patch(msg_1M, v_direct), 2))      # expect ~3.54

# Path 2: the same thing through the dict + intervene() (what cell 10 did)
ch = {int(i): f6[int(i)].item() for i in np.where(change > 0)[0]}
f_new = f1.clone()
for k_, v_ in ch.items():
    f_new[k_] = v_
print("n changes:", len(ch))
print("max |f_new - f6|:", (f_new - f6).abs().max().item())               # expect 0
print("D via intervene:", round(intervene(msg_1M, ch), 2))                # should match v_direct

max |v_direct - C|: 3.814697265625e-06
D with v_direct: 3.54
n changes: 100
max |f_new - f6|: 0.0
D via intervene: 3.54


#### More samples:

In [15]:
rng = np.random.default_rng(SEED)

# use sample with a clear 1M->6M effects (>0.3, small effects make recovery unstable)
eff = ft["d6"] - ft["d1"]
use = np.where(eff > 0.3)[0]
print(f"Samples with 1M->6M effect > 0.3: {len(use)} / {len(eff)}")

selected = [int(x) for x in list(top_positive) + list(top_negative)]
K_LIST = [1, 5, 10, 20, 50]
rows = []   # one dict per sample

for s_ in tqdm(use):
    ex = test_dataset[int(cf_idx[s_])]
    m1 = make_hno3_counterfactual(ex, 1)
    m6 = make_hno3_counterfactual(ex, 6)
    a1_, dd1 = run_model(m1)
    a6_, dd6 = run_model(m6)
    g1, g6 = encode(a1_), encode(a6_)

    ch_ = (g6 - g1).abs().cpu().numpy()
    changed = np.where(ch_ > 0)[0]
    order_ = changed[np.argsort(ch_[changed])[::-1]]       # changed features, biggest change first

    # 1M run with these features set to their 6M values -> recovery (0..1)
    def rec_(ids):
        d = intervene(m1, {int(i): g6[int(i)].item() for i in ids})
        return (d - dd1) / (dd6 - dd1)

    r = {f"top{k}": rec_(order_[:k]) for k in K_LIST}
    r["selected20"] = rec_(selected)
    r["random20"] = rec_(rng.choice(changed, size=min(20, len(changed)), replace=False))
    r["overlap"] = len(set(order_[:20]) & set(selected))   # selected features among top-20 changed
    r["rank1208"] = int(np.where(order_ == REP_HIGH)[0][0]) + 1 if REP_HIGH in order_ else None
    rows.append(r)

import pandas as pd
rec_df = pd.DataFrame(rows)
print(rec_df.drop(columns=["rank1208"]).median().round(2))

Samples with 1M->6M effect > 0.3: 401 / 458


  0%|          | 0/401 [00:00<?, ?it/s]

top1          0.13
top5          0.40
top10         0.60
top20         0.76
top50         0.89
selected20    0.39
random20      0.15
overlap       4.00
dtype: float64


In [16]:
print("Median recovery (fraction of real 1M->6M effect):")
for col in [f"top{k}" for k in K_LIST] + ["selected20", "random20"]:
    print(f"  {col:<11} {rec_df[col].median():5.0%}   (IQR {rec_df[col].quantile(.25):.0%} – {rec_df[col].quantile(.75):.0%})")

print(f"\nSelected features among each sample's top-20 changed: median {rec_df['overlap'].median():.0f} / 20")
r1208 = rec_df["rank1208"].dropna()
print(f"Feature {REP_HIGH} changed in {len(r1208)}/{len(rec_df)} samples; "
      f"median rank when changed = {r1208.median():.0f}, top-5 in {(r1208 <= 5).mean():.0%} of them")

Median recovery (fraction of real 1M->6M effect):
  top1          13%   (IQR 8% – 23%)
  top5          40%   (IQR 31% – 59%)
  top10         60%   (IQR 48% – 76%)
  top20         76%   (IQR 67% – 87%)
  top50         89%   (IQR 82% – 95%)
  selected20    39%   (IQR 29% – 54%)
  random20      15%   (IQR 7% – 23%)

Selected features among each sample's top-20 changed: median 4 / 20
Feature 1208 changed in 320/401 samples; median rank when changed = 3, top-5 in 66% of them


**Feature-level patching (401 test samples with 1 M → 6 M effect > 0.3)**

For each sample, SAE features of the 1 M run were replaced with their 6 M values, and recovery was measured as the fraction of the real 1 M → 6 M change in expected D.

| Features patched | Median recovery (IQR) |
|---|---|
| Top 1 most-changed | 13% (8–23%) |
| Top 5 | 40% (31–59%) |
| Top 10 | 60% (48–76%) |
| Top 20 | 76% (67–87%) |
| Top 50 | 89% (82–95%) |
| 20 D-correlated features (from 03) | 39% (29–54%) |
| 20 random changed features | 15% (7–23%) |

- The HNO₃ effect is carried by a small set of features: about 20 of 2048 features account for three quarters of it, far more than 20 randomly chosen changed features.
- Feature 1208 changes in 80% of samples and is usually among the top 3, but alone it accounts for only part of the effect.
- Features selected in 03 by correlation with D carry part of the effect (39%), but overlap only partly with the most-changed features (median 4 of 20). Correlation with D and causal use for the HNO₃ response are related but not the same.

In [17]:
# Per-sample patching results
rec_df.to_csv("/kaggle/working/feature_patching_recovery.csv", index=False)

# Base vs ft comparison (cell 6) and the sample indices used
np.savez_compressed(
    "/kaggle/working/base_vs_ft.npz",
    cf_idx=cf_idx, use=use,
    ft_d1=results["ft"]["d1"], ft_d6=results["ft"]["d6"],
    base_d1=results["base"]["d1"], base_d6=results["base"]["d6"],
    ft_rq=results["ft"]["rq"], base_rq=results["base"]["rq"],
)
print("Saved.")

Saved.
